<a href="https://colab.research.google.com/github/guirco/ufpel-pdi-2026-1/blob/main/LAB10_Segmentacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB10 — Segmentação de Imagens

Disciplina: **Processamento Digital de Imagens (PDI)** – UFPel  
Professor: **Guilherme Corrêa**  

Vamos utilizar a imagem `leucocitos.jpg` como exemplo nos exercícios.

---

## **Objetivo Geral**

**O seu código deve encontrar os leucócitos na imagem e contá-los.**
Para auxiliar a análise morfológica manual, neste trabalho você deverá desenvolver um *script* que permita identificar e contar instâncias de leucócitos a partir de imagens de entrada que representam células sanguíneas. Os leucócitos são representados por regiões mais escuras (roxo escuro) e os glóbulos vermelhos são representados por regiões mais claras (roxo claro, vermelho fraco ou rosa).

## **Contexto**

A contagem e análise de células sanguíneas permite a avaliação e diagnóstico de um grande número de doenças, como leucemia e linfocitose. Em particular, a análise de leucócitos (também conhecidos como células brancas ou glóbulos brancos) é um tópico de grande interesse para hematologistas. Atualmente, na maioria dos casos, a análise e a contagem de leucócitos em imagens do sangue obtidas por microscópio são realizadas manualmente por hematologistas.

## **Dicas**
1. Você poderá usar quaisquer bibliotecas e funções que julgar necessárias.
2. Você pode usar algoritmos de segmentação, limiarização, processamento morfológico, conversão entre espaços de cores, filtros de média e mediana, etc.
3. A implementação e a entrega é individual, mas você pode trocar ideias com os colegas.
4. Filtro de mediana no OpenCV: `cv2.medianBlur(img, kernel)`
5. Limiarização com método de Otsu: `filters.threshold_otsu(img)`

## Bibliotecas úteis
Se estiver no Colab, rode a célula de instalação uma única vez.

In [ ]:
# Se estiver no Colab, descomente a linha abaixo para garantir o OpenCV instalado
# !pip install opencv-python

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from google.colab import files

## Upload de uma imagem

Vamos usar `files` do `google.colab` para fazer upload de uma imagem colorida e a biblioteca OpenCV (`cv2`) para abri-la.

Sugestão: utilizar a imagem `leucocitos.jpg`, que está disponível no repositório para os exercícios. A imagem está no formato RGB.

In [ ]:
print("Faça upload de uma imagem (JPG/PNG).")
up = files.upload()
if not up:
    raise RuntimeError("Nenhum arquivo enviado.")

# Nome do arquivo
fname = next(iter(up))

# Ler imagem colorida (BGR)
img_rgb = cv2.imdecode(np.frombuffer(up[fname], np.uint8), cv2.IMREAD_COLOR)
if img_rgb is None:
    raise RuntimeError("Falha ao ler a imagem.")

# Mostrar
plt.imshow(img_rgb, vmin=0, vmax=255)
plt.axis('off')
plt.title('Imagem Colorida')
plt.show()

# 🖼️ Tarefa A: Visualizar a imagem no formato RGB e aplicar a filtragem

1. Trocar os canais R e B na imagem para poder visualizar os canais de cores corretas.
2. Exibir a imagem corrigida, junto com os canais individualmente.

```
cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
```
3. Aplicar o filtro de mediana (kernel=17x17) à imagem original.
4. Exibir a imagem filtrada, junto com os canais individualmente.
```
img_med = cv2.medianBlur(img_rgb, 17)
```
5. Observar a diferença entre a imagem original e a filtrada.

In [20]:
from skimage import io, filters, measure
# SEUS CÓDIGOS AQUI

# 🖼️ Tarefa B: Preparando imagem binarizada inicial

1. Selecionar o canal B e G
3. Calcular o canal G menos o canal B e exiba-la
4. Aplicar o método de Otsu para encontrar o threshold ideal para separar glóbulos brancos e glóbulos vermelhos
```
filters.threshold_otsu(mR)
```
5. Rotular com `measure.label(1-thresh, connectivity=2)` (é necessário calcular 1-thresh, pois a imagem thresh é formada com fundo branco e não com fundo preto)
6. Calcular o número de regiões encontradas considerando vizinhança-8 (conectividade=2)

In [21]:
# SEUS CÓDIGOS AQUI

In [ ]:
# label
labels = measure.label(1-thresh, connectivity=2)

num_regioes = labels.max()
print("Regiões encontradas:", num_regioes)

Regiões encontradas: 27


# Plotando imagem binarizada

A seguir, estamos apenas plotando a imagem após limiarização.

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(thresh, cmap='gray')
plt.axis('off')
plt.show()

# Funções: Dilatação e Erosão

A seguir estamos:
1. Implementando função de Dilatação
2. Implementando função de Erosão

In [ ]:
def dilatacao(img, kernel):
    h, w = img.shape
    kh, kw = kernel.shape
    pad_h = kh // 2
    pad_w = kw // 2

    # imagem de saída inicializada com zeros
    out = np.zeros_like(img)

    # aplica padding
    padded = np.pad(img, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant')

    for i in range(h):
        for j in range(w):
            # extrai vizinhança
            region = padded[i:i+kh, j:j+kw]

            # se qualquer pixel da região onde o kernel é 1 for branco, o pixel vira branco
            if np.any(region[kernel == 0] == 0):
                out[i, j] = 0
            else:
                out[i, j] = 255

    return out

In [ ]:
def erosao(img, kernel):
    h, w = img.shape
    kh, kw = kernel.shape
    pad_h = kh // 2
    pad_w = kw // 2

    out = np.zeros_like(img)

    padded = np.pad(img, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant')

    for i in range(h):
        for j in range(w):
            region = padded[i:i+kh, j:j+kw]

            # se todos os pixels da região onde kernel=1 forem brancos, mantém branco
            if np.all(region[kernel == 0] == 0):
                out[i, j] = 0
            else:
                out[i, j] = 255

    return out

# 🖼️ Tarefa C: Aplicando Erosão e Dilatação

1. Aplicar erosão com kernel 25x25 para eliminar ruídos remanscentes.
2. Depois, aplicar dilatação para restaurar tamanho original dos elementos restantes.

In [22]:
# SEUS CÓDIGOS AQUI

In [23]:
# SEUS CÓDIGOS AQUI

# Encontrando regiões conexas (conectividade-8)

In [ ]:
# encontrando regiões

labels = measure.label(dil==0, connectivity=2)

num_regioes = labels.max()
print("Regiões encontradas:", num_regioes)

In [ ]:
# rotulando regiões na imagem final

props = measure.regionprops(labels)

fig, ax = plt.subplots()
ax.imshow(dil, cmap='gray')
ax.axis('off')

for i, region in enumerate(props, start=1):
    # centróide da região (linha = y, coluna = x)
    y, x = region.centroid

    # desenha o número na imagem
    ax.text(x, y, str(i),
            color='white', fontsize=8, ha='center', va='center')